# Spark Exercise

Apache Spark is an excellent tool for data engineering projects due to its robust ability to process large-scale data efficiently through distributed computing. Spark's in-memory processing capabilities significantly enhance the speed of data operations, making it ideal for handling big data workloads. It supports various data sources and formats, offering versatility in data ingestion and transformation. Additionally, Spark's rich API supports multiple programming languages such as Python, Java, and Scala, catering to diverse developer preferences. Its ecosystem, which includes libraries for SQL, machine learning, and graph processing, provides a comprehensive suite for building complex data pipelines and analytics, making it a powerful and flexible choice for data engineering tasks.

Use Python, ```pyspark``` and ```pandas``` to explore Apache Spark RDD and DataFrame:

# Spark RDD

Spark RDD (Resilient Distributed Dataset) is a fundamental data structure in Apache Spark that enables fault-tolerant, distributed processing of large datasets across multiple nodes in a cluster. Spark RDDs provide a higher-level abstraction for performing distributed data processing tasks, including both map (transformations) and reduce (aggregations) operations.

## Import Necessary Libraries

In [1]:
import csv
from io import StringIO

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

## Spark Context and Session
Initialize Spark Context and Spark Session

In [2]:
spark = (SparkSession
    .builder
    .master("local")
    .appName("bdeng-spark")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(sc.version)
print(sc.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 23:57:30 WARN Utils: Your hostname, LaggerThinkPad, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/09 23:57:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 23:57:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.2
local


## Load Data into RDD

In [3]:
csv_file = "cleaned-amazon-products-electronics-sales-2023.csv"

lines_rdd = sc.textFile(csv_file)
header = lines_rdd.first()

def is_not_header(line):
    return line != header

data_lines_rdd = lines_rdd.filter(is_not_header)

def parse_csv_line(line):
    line_as_file = StringIO(line)
    csv_reader = csv.reader(line_as_file)
    row = next(csv_reader)
    return row

products_rdd = data_lines_rdd.map(parse_csv_line)

# This prints the number of products that are in the RDD.
print(products_rdd.count())

9411


## Map Operation

Split data into individual parts and create key-value pairs

In [4]:
def get_rating_pair(row):
    rounded_rating = round(float(row[5]))
    return rounded_rating, 1

rating_pairs_rdd = products_rdd.map(get_rating_pair)

# This prints the first 10 key value pairs (rating, counter).
print(rating_pairs_rdd.take(10))

[(4, 1), (4, 1), (4, 1), (4, 1), (4, 1)]


## Reduce Operation

Reduce your key-value pairs

In [5]:
def add_numbers(first_number, second_number):
    return first_number + second_number

rating_counts_rdd = rating_pairs_rdd.reduceByKey(add_numbers)

## Collect Results

Because of lazy evaluation, the map-reduce operation is performed only now. Show what you calculated.

In [6]:
rating_counts = rating_counts_rdd.collect()

def sort_by_rating(item):
    rating = item[0]
    return rating

rating_counts = sorted(rating_counts, key=sort_by_rating)

# This prints the number of Amazon products that have this rating.
for rating, product_count in rating_counts:
    print(rating, product_count)

1 10
2 31
3 473
4 8406
5 491


## Save Results

In [7]:
rating_counts_rdd.saveAsTextFile("saved-rating-rdd")
print("RDD saved.")

RDD saved.


# Spark DataFrame

Spark DataFrame is a distributed collection of data organized into named columns, designed for efficient data manipulation and analysis in Apache Spark. It is used for various data processing tasks such as data ingestion, transformation, querying, and analysis in Apache Spark, providing a high-level abstraction that simplifies working with structured data.

## Load Data into DataFrame

In [8]:
schema = StructType([
    StructField("name", StringType(), True),
    StructField("main_category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("image", StringType(), True),
    StructField("link", StringType(), True),
    StructField("ratings", DoubleType(), True),
    StructField("no_of_ratings", IntegerType(), True),
    StructField("discount_price", DoubleType(), True),
    StructField("actual_price", DoubleType(), True),
])

df = (
    spark.read
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    .schema(schema)
    .csv(csv_file)
)

## View DataFrame Schema

In [9]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- image: string (nullable = true)
 |-- link: string (nullable = true)
 |-- ratings: double (nullable = true)
 |-- no_of_ratings: integer (nullable = true)
 |-- discount_price: double (nullable = true)
 |-- actual_price: double (nullable = true)



## View DataFrame Data

In [10]:
df.createOrReplaceTempView("amazon_products")

products_sample_df = spark.sql("""
                               SELECT *
                               FROM amazon_products
                                        LIMIT 10
                               """)

# This shows the first 10 rows of the DataFrame.
products_sample_df.show()

+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|Redmi 10 Power (P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.0|          965|        120.99|      208.99|
|OnePlus Nord CE 2...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.3|       113956|        208.99|      219.99|
|OnePlus Bullets Z...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.2|        90304|         21.99|       25.29|
|Samsung Galaxy M3...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.ama

In [1]:
df.show(10)

NameError: name 'df' is not defined

## Filter Data

Performe a filter operation on a column

In [12]:
cheap_products_df = spark.sql("""
                              SELECT *
                              FROM amazon_products
                              WHERE discount_price < 20.0
                              """)

# This prints the amount of products with a discount price < 20 EUR.
print(cheap_products_df.count())

# This shows the first 10 products with a discount price < 20 EUR.
cheap_products_df.show(10)

7349
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.9|       172347|         14.29|       43.94|
|Apple 20W USB-C P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.6|        61748|         17.48|        20.9|
|boAt Airdopes Ato...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.8|         3327|         14.29|       49.39|
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://ww

In [13]:
cheap_products_df = df.filter(df.discount_price < 20.0)

print(cheap_products_df.count())
cheap_products_df.show(10)

7349
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.9|       172347|         14.29|       43.94|
|Apple 20W USB-C P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.6|        61748|         17.48|        20.9|
|boAt Airdopes Ato...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.8|         3327|         14.29|       49.39|
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://ww

## Group By and Aggregate

Performe a group by and aggregat operation

In [14]:
rating_summary_df = spark.sql("""
                              SELECT
                                  ratings,
                                  COUNT(*) AS product_count,
                                  AVG(discount_price) AS avg_discount_price
                              FROM amazon_products
                              GROUP BY ratings
                              ORDER BY ratings
                              """)

# This shows the first 10 ratings with the amount of products that have this rating and the average discount price for this rating.
rating_summary_df.show(10)

+-------+-------------+------------------+
|ratings|product_count|avg_discount_price|
+-------+-------------+------------------+
|    1.0|            8|           6.63125|
|    1.3|            1|              7.69|
|    1.4|            1|              5.82|
|    1.5|            2|             9.285|
|    1.7|            2|             13.69|
|    1.8|            1|              2.19|
|    1.9|            1|              3.58|
|    2.0|            2|             2.905|
|    2.1|            3|              5.53|
|    2.2|            4|             5.255|
|    2.3|            3| 9.486666666666666|
|    2.4|            6|14.295000000000002|
|    2.5|            7| 6.029999999999999|
|    2.6|            7| 84.44714285714285|
|    2.7|            7| 7.817142857142857|
|    2.8|           18|19.789999999999996|
|    2.9|           25|14.839199999999998|
|    3.0|           43|30.374186046511625|
|    3.1|           44| 26.37636363636364|
|    3.2|           58| 16.92827586206897|
+-------+--

In [15]:
grouped_by_rating_df = df.groupBy(df.ratings)

rating_summary_df = grouped_by_rating_df.agg(
    count("*").alias("product_count"),
    avg(df.discount_price).alias("avg_discount_price"),
)

rating_summary_df = rating_summary_df.sort("ratings")

rating_summary_df.show()

+-------+-------------+------------------+
|ratings|product_count|avg_discount_price|
+-------+-------------+------------------+
|    1.0|            8|           6.63125|
|    1.3|            1|              7.69|
|    1.4|            1|              5.82|
|    1.5|            2|             9.285|
|    1.7|            2|             13.69|
|    1.8|            1|              2.19|
|    1.9|            1|              3.58|
|    2.0|            2|             2.905|
|    2.1|            3|              5.53|
|    2.2|            4|             5.255|
|    2.3|            3| 9.486666666666666|
|    2.4|            6|14.295000000000002|
|    2.5|            7| 6.029999999999999|
|    2.6|            7| 84.44714285714285|
|    2.7|            7| 7.817142857142857|
|    2.8|           18|19.789999999999996|
|    2.9|           25|14.839199999999998|
|    3.0|           43|30.374186046511625|
|    3.1|           44| 26.37636363636364|
|    3.2|           58| 16.92827586206897|
+-------+--

## Save DataFrame to Parquet

In [16]:
df.write.parquet("saved-products-dataframe")
print("DataFrame saved.")

DataFrame saved.


In [17]:
spark.stop()